In [64]:
import h5py
import numpy
import re
from collections import defaultdict

SPLIT_FILENAME = re.compile(r"_(SPLIT\d+)")

NETWORK_IFO_EVENT_ID_REGEX = re.compile(
    r"\Anetwork/(?P<ifo>[A-Z]1)_event_id\Z",
)
EVENT_ID_REGEX = re.compile(r"event_id\Z")

def read_hdf5_triggers(inputfiles, verbose=False):
    """Load several HDF5 trigger files into a single dictionary.

    Parameters
    ----------
    inputfiles : `list` of `str`
        the paths of the input HDF5 files to merge
    """
    datasets = {}

    def _scan_dataset(name, obj):
#         if not name.startswith("network") or not isinstance(obj, h5py.Dataset):
#             return
        if not isinstance(obj, h5py.Dataset):
            return
#         if NETWORK_IFO_EVENT_ID_REGEX.match(name):
#             return
        shape = obj.shape
        dtype = obj.dtype
        try:
            shape = numpy.sum(datasets[name][0] + shape, keepdims=True)
        except KeyError:
            pass
        else:
            assert dtype == datasets[name][1], (
                "Cannot merge {0}/{1}, does not match dtype".format(
                    obj.file.filename, name,
                ))
        datasets[name] = (shape, dtype)

    # get list of datasets
    datasets = {}
    for filename in inputfiles:
        with h5py.File(filename, 'r') as h5f:
            h5f.visititems(_scan_dataset)

    position = defaultdict(int)
    ifo_eventid_increment = defaultdict(int)
    
    out = defaultdict(lambda: defaultdict(dict))

    # create datasets
    for dset, (shape, dtype) in datasets.items():
        prefix = dset.split("/")[0]
        key = dset[len(prefix)+1:]
        out[prefix][key] = numpy.empty(shape, dtype=dtype)

    # copy dataset contents
    for filename in inputfiles:
        with h5py.File(filename, 'r') as h5in:
            for dset in datasets:
                data = h5in[dset][:]
                size = data.shape[0]
                pos = position[dset]
                prefix = dset.split("/")[0]
                key = dset[len(prefix)+1:]
                
                if EVENT_ID_REGEX.search(dset):
                    if NETWORK_IFO_EVENT_ID_REGEX.search(dset):
                        # must increment eventids differently
                        ifo_eid_key = "{}/{}_{}".format(*dset[8:].split("_"))
                        ifo_eid_incr = ifo_eventid_increment[ifo_eid_key]
                        out[prefix][key][pos:pos+size] = data + ifo_eid_incr
                        ifo_eid_size = h5in[ifo_eid_key].shape[0]
                        ifo_eventid_increment[ifo_eid_key] += ifo_eid_size
                    else:
                        out[prefix][key][pos:pos+size] = data + pos
                else:
                    out[prefix][key][pos:pos+size] = data
                position[dset] += size
    return dict(out)

In [65]:
trigfiles = ["./test9.hdf", "./test9.hdf"]
triggers = read_hdf5_triggers(trigfiles)

In [108]:
ifo_list = [k for k in triggers.keys() if k != "network"]
ifo_times = {}
for ifo in ifo_list:
    net_ifo_eid = triggers["network"]["{}_event_id".format(ifo)]
    ifo_times[ifo] = triggers[ifo]["end_time"][net_ifo_eid]
mean_time = [events.mean_if_greater_than_zero(v)[0] 
             for v in numpy.array(list(ifo_times.values())).T]
triggers["network"]["mean_end_time"] = mean_time

In [122]:
for col in triggers:
    print(col)

H1
L1
V1
network


In [ ]:
found.append(i)
# r-l == 1 means the injection has exactly one event 
# associated, while r-l>1 indicates more than one
# was associated within the window.
eid = l if r - l == 1 else event_id[l + snr[l:r].argmax()]
##### FIXME ##########
for col in triggers:
    trigs[col].append(triggers[col][i])

In [124]:
triggers["H1"]

defaultdict(dict,
            {'coa_phase_dom': array([0., 0., 0., 0.], dtype=float32),
             'coa_phase_sub': array([0., 0., 0., 0.], dtype=float32),
             'end_time': array([1234.00195312, 1234.00585938, 1234.00195312, 1234.00585938]),
             'event_id': array([0, 1, 2, 3]),
             'search/end_time': array([1235, 1235], dtype=int32),
             'search/start_time': array([1234, 1234], dtype=int32),
             'sigmasq_dom': array([100., 100., 100., 100.], dtype=float32),
             'sigmasq_sub': array([50., 50., 50., 50.], dtype=float32),
             'snr_dominant_h1': array([ 6., 18.,  6., 18.], dtype=float32),
             'snr_subdominant_h1': array([ 6., 18.,  6., 18.], dtype=float32),
             'template_duration': array([1., 1., 1., 1.], dtype=float32),
             'template_hash': array([123, 123, 123, 123]),
             'time_index': array([1, 3, 1, 3])})

# Where to calculate the mean gps time:

* Put the above lines in write_to_hdf.

* Possibly also put it in the read_hdf5 function; I want to also load the individual ifo stuff.